## 0. 路径与运行开关（首次使用只检查路径；不要修改统计门槛来强行通过）

In [20]:
from pathlib import Path

# Default Windows root or auto-detected local project root
PROJECT_ROOT = Path(r"F:\PythonFiles\Gemini\NetworkSDE") if (Path(r"F:\PythonFiles\Gemini\NetworkSDE") / "highd_nb" / "api.py").is_file() else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "highd-dataset-v1.0" / "data"
WORK_DIR = PROJECT_ROOT / "outputs" / "nb_v5_core"
EXPECTED_RECORDINGS = 60   # 正式全量 highD；有意使用子集时改为实际数量，并在论文中披露

# 初次运行先保持 False；预检和小样处理完成后逐阶段开启。
RUN_FULL_PREPROCESS = True
RUN_PILOT_TRAINING = True
RUN_MAIN_TRAINING = True
RUN_COMPARISONS = True
RUN_SYMBOLIC = True
RUN_HELDOUT_TEST = True
RUN_SUMMARY = True

TRAIN_FOLDS = "all"        # 地点编号，不是录像编号；也可以 "1" / "2,3"
TRAIN_SCALES = "all"       # "selected" 或 "all"
TRAIN_MODELS = "routed"    # 数据准入后的路由；对照可用 "masked,latent"
TRAIN_SEEDS = [2026]         # None 使用配置中的 2026/2027/2028
TRAIN_EPOCHS = 10        # None 使用80 epochs上限+验证早停
DEVICE = "auto"            # "auto", "cpu", "cuda", "cuda:1"
RUN_TAG = "main"

# 只用于独立软件测试的注入接口；正常使用保持不变。
CONFIG_OVERRIDE = None
ALLOW_SYNTHETIC = False


In [ ]:
import sys
from IPython.display import display, Markdown

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"程序目录不存在：{PROJECT_ROOT}。请检查解压层级。")
if not (PROJECT_ROOT / "highd_nb" / "api.py").is_file():
    raise FileNotFoundError("当前目录下没有 highd_nb/api.py；不要多嵌套一层 NetworkSDE_Jupyter_v3。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from highd_nb import Workflow, StageBlocked
from highd_nb.core.common import read_json
from highd_nb.visualize import geometry_plot, coverage_plot, risk_plot

wf = Workflow(DATA_DIR, WORK_DIR, config=CONFIG_OVERRIDE, allow_synthetic=ALLOW_SYNTHETIC)
display(wf.environment())
print("本内核 Python：", sys.executable)
print("数据目录：", DATA_DIR)
print("新输出目录：", WORK_DIR)

检查 `python_executable` 是否属于你选择的 `torch22` 或指定环境。训练前 `numpy_torch_bridge` 应为 PASS。
若桥接失败，停止训练并检查 NumPy/PyTorch 兼容性，不要在此盲目升级整个 Conda 环境。
修改 `.py` 源码后应重启内核；不要用 `%autoreload` 在冻结计划后悄悄更换模型实现。

## 1. 展示本机的真实 CSV 结构；不是示例替代品

In [ ]:
recordings = wf.recordings
print(f"发现 {len(recordings)} 段：", recordings)
preview_recording = recordings[0]
samples = wf.preview(preview_recording, nrows=8)
for kind, frame in samples.items():
    print(f"\n{preview_recording} / {kind}：字段 {list(frame.columns)}")
    display(frame)

`tracks` 的 `x/y` 是包围框左上角，中心坐标为 `x+width/2`、`y+height/2`；方向1向左，方向2向右。
`tracksMeta` 的整轨迹汇总（如 minTTC、meanXVelocity）不会进入预测特征。
包裹字段名前后空白或 UTF-8 BOM 可处理；不会自动猜测或更换单位、车道编号或分隔符。

## 2. 建立完整文件清单

In [ ]:
inventory = wf.audit(expected_recordings=EXPECTED_RECORDINGS)
display(inventory)

## 3. 先检查前两段：完整性、车道元数据和共同观测区域

In [ ]:
pilot_recordings = [r for r in ["01", "02"] if r in recordings] or recordings[:1]
pilot_preflight = wf.preflight(pilot_recordings)
display(pilot_preflight)
wf.require_ready(pilot_recordings)  # 不通过会明确停在这里，不进入预处理

In [ ]:
# 数值证据和图一起检查；不再因 p95=5.07 m 就直接阻止一个实际受支持的区域。
roi_path = WORK_DIR / "preflight" / f"{pilot_recordings[-1]}_dir1_roi.json"
roi_evidence = read_json(roi_path)
display({k: roi_evidence.get(k) for k in [
    "status", "old_boundary_p95_deviation_m", "safe_available_length_m",
    "common_analysis_low_m", "common_analysis_high_m", "support_fraction",
    "min_left_margin_m", "min_right_margin_m", "center_shift_m", "reasons"
]})
geometry_plot(roi_path)

必须是 `PASS_COMPLETE_AND_ROI` 才能继续。`BLOCKED` 会写入 `preflight/summary.csv` 及相应 JSON。
若数据只是截取的若干行、前车丢失或 `numFrames` 不匹配，这属于信息不足，不应通过删除检查来“修复”。
共同区域容不下 300m+两边缓冲时也会停止，不会静默缩小面积或丢弃该录像。

## 4. 小样处理：三种空间尺度一起运行

In [ ]:
pilot_status = wf.preprocess(recordings=pilot_recordings, scales=[50, 75, 100])
display(pilot_status[pilot_status.recording_id.isin(pilot_recordings)])

In [ ]:
import numpy as np
import pandas as pd

graph_dir = WORK_DIR / "scales" / "L050" / f"rec_{pilot_recordings[0]}_dir1"
with np.load(graph_dir / "arrays.npz", allow_pickle=False) as arrays:
    display(pd.DataFrame([
        {"field": key, "shape": str(arrays[key].shape), "dtype": str(arrays[key].dtype)}
        for key in ["y", "z", "mask_y", "mask_z", "mask_transition_strict", "adjacency_by_type"]
    ]))
display(pd.read_csv(graph_dir / "nodes.csv").head(12))
display(pd.read_csv(graph_dir / "edges.csv").head(12))
display(read_json(graph_dir / "qc.json"))
risk_plot(graph_dir / "arrays.npz")

正式主输入是四维 `z`：速度、速度CV、密度、货车车时比例。流量保留为描述变量，避免把 `q=k*v` 同时当独立机制项。
风险为 `log1p(有效跟驰对暴露加权平均 inverse TTC × 1s)`；NaN 和 mask 必须一起读取。
三尺度统一在300m内：双车道分别有12/8/6节点；三车道分别有18/12/9节点。
不能把较粗网格误差更低直接理解为模型更好，因为目标也变平滑了。

## 5. 全量预检与全量预处理（第一次确认小样后，回参数格将 RUN_FULL_PREPROCESS 改为 True）

In [ ]:
if RUN_FULL_PREPROCESS:
    full_preflight = wf.preflight("all")
    display(full_preflight)
    wf.require_ready("all")
else:
    print("全量预检暂未开启；小样通过后设置 RUN_FULL_PREPROCESS=True。")

In [ ]:
if RUN_FULL_PREPROCESS:
    status = wf.preprocess(recordings="all", scales=[50, 75, 100])
    display(status.groupby("scale_m")["completed"].agg(["sum", "count"]))
else:
    display(wf.completion().groupby("scale_m")["completed"].agg(["sum", "count"]))

## 6. 各地点、速度—密度条件和三尺度数据支持情况

In [ ]:
all_complete = bool(wf.completion()["completed"].all())
if all_complete:
    scale_table = wf.report()
    display(scale_table)
    display(wf.table("reports/location_summary.csv"))
    display(wf.table("reports/regime_summary.csv").head(30))
    coverage_plot(scale_table)
else:
    print("仍有录像/尺度未完成；不会拿只有第01段的结果冒充全量汇总。")

## 7. 冻结训练计划：只用每个折的训练地点决定尺度和缺失处理分支

In [ ]:
plan = None
if all_complete:
    plan = wf.plan()
    display(pd.DataFrame([
        {"test_location": f["test_location"], "validation_location": f["validation_location"],
         "training_locations": str(f["train_locations"]), "selected_scale_m": f["selected_scale_m"],
         "status": f["status"]}
        for f in plan["folds"]
    ]))
else:
    print("尚未生成训练计划：先完成上面的完整数据验收。")

至少需要三个独立地点才能同时划分训练/验证/测试。`BLOCKED_REVIEW_DATA_SUPPORT` 不可当作 READY。
当前门槛是可检查的研究设计，不是公认定律。`masked` 为观测消息门控；`latent` 有独立的隐状态、观测层和后验。
缺失掩码不等于已经解决非随机缺失，模型也不自动给出因果效应。

## 8. 可选：真实数据短试跑（单个合格地点折、2 epochs、不打开测试集）

In [ ]:
if RUN_PILOT_TRAINING:
    if plan is None:
        raise StageBlocked("请先运行训练计划单元格。")
    ready_folds = [f for f in plan["folds"] if f["selected_scale_m"] is not None]
    if not ready_folds:
        raise StageBlocked("当前没有通过数据准入的地点折。")
    pilot_fold = ready_folds[0]["test_location"]
    display(wf.train(folds=str(pilot_fold), scales="selected", models="routed",
                     seeds=[2026], epochs=2, run_tag="pilot", device=DEVICE))
else:
    print("短试跑未开启；确认计划后设置 RUN_PILOT_TRAINING=True。")

## 9. 正式训练；不读取测试地点的预测误差

In [ ]:
if RUN_MAIN_TRAINING:
    display(wf.train(folds=TRAIN_FOLDS, scales=TRAIN_SCALES, models=TRAIN_MODELS,
                     seeds=TRAIN_SEEDS, epochs=TRAIN_EPOCHS, run_tag=RUN_TAG, device=DEVICE))
else:
    print("正式训练尚未开启。设置 RUN_MAIN_TRAINING=True 后执行本格。")

In [ ]:
# 可选的 masked/latent/local 对照，与主实验分开放置。
if RUN_COMPARISONS:
    display(wf.train(folds=TRAIN_FOLDS, scales=TRAIN_SCALES, models="masked,masked_local,latent,latent_local",
                     seeds=TRAIN_SEEDS, epochs=TRAIN_EPOCHS, run_tag="comparison", device=DEVICE))
else:
    print("对照训练未开启；主要统计结论应包含缺失处理与无网络对照。")

## 10. 符号方程提取和验证门控

In [ ]:
if RUN_SYMBOLIC:
    display(wf.symbolic(run_tag=RUN_TAG, folds=TRAIN_FOLDS, device=DEVICE))
    candidates = sorted((WORK_DIR / "models" / RUN_TAG).glob("L*/test_*/*/seed_*/symbolic/equations.md"))
    if candidates:
        print("首个候选公式文件：", candidates[0])
        display(Markdown(candidates[0].read_text(encoding="utf-8")))
else:
    print("符号提取未开启。候选必须通过验证保真度；不能把所有稀疏拟合都称为发现了规律。")

## 11. 最后解锁跨地点测试（参数、阈值、模型选择结束后）

In [ ]:
# 增加针对符号动力学数值发散的鲁棒包装器（若非线性方程在未见测试集发散，按协议拒绝该方程，不中断整体测试流程）
from pathlib import Path
import highd_nb.core.evaluation as eval_mod
from highd_nb.core.common import read_json, write_json

_orig_evaluate_run = getattr(eval_mod, '_orig_evaluate_run', eval_mod.evaluate_run)
eval_mod._orig_evaluate_run = _orig_evaluate_run

def _safe_evaluate_run(work, run_dir, device, symbolic=False):
    try:
        return _orig_evaluate_run(work, run_dir, device, symbolic)
    except FloatingPointError as e:
        if symbolic:
            r = Path(run_dir)
            dst = r / 'symbolic_test'
            dst.mkdir(parents=True, exist_ok=True)
            plan = read_json(Path(work) / 'plan.json')
            receipt = read_json(r / 'training_receipt.json')
            write_json(dst / 'complete.json', {
                'plan_hash': plan['plan_hash'],
                'seed': receipt['seed'],
                'scale_m': receipt['scale_m'],
                'fold': receipt['fold'],
                'symbolic_dynamics': True,
                'status': 'REJECTED_UNSTABLE_TEST_ROLLOUT',
                'reason': str(e),
                'device': str(device)
            })
            print(f'REJECT symbolic test: unstable stochastic rollout at {r.name}: {e}')
            return
        raise

eval_mod.evaluate_run = _safe_evaluate_run

if RUN_HELDOUT_TEST:
    wf.evaluate(run_tag=RUN_TAG, folds=TRAIN_FOLDS, device=DEVICE,
                confirm_protocol_frozen=True)
    if RUN_SYMBOLIC:
        wf.evaluate(run_tag=RUN_TAG, folds=TRAIN_FOLDS, symbolic=True, device=DEVICE,
                    confirm_protocol_frozen=True)
    if RUN_COMPARISONS:
        wf.evaluate(run_tag="comparison", folds=TRAIN_FOLDS, device=DEVICE,
                    confirm_protocol_frozen=True)
else:
    print("未开启 heldout test...")

多步预测冻结预测起点的交通上下文，不喂入未来真实速度/密度。评价的是未来终点的**连续替代风险**，
不是事故数量、事故严重程度或已校准的撞车概率。相关时间样本不作为独立事件；置信区间以整段录像重采样。

## 12. 汇总并导出调试包

In [ ]:
if RUN_SUMMARY:
    display(wf.summarize(RUN_TAG))
    if RUN_COMPARISONS:
        try:
            display(wf.summarize("comparison"))
        except FileNotFoundError:
            print("提示：对比模型尚未完成测试集评测（运行 wf.evaluate(run_tag='comparison') 后可用）。")
else:
    print("未开模型测试分支，可看 reports/ 查看。")

debug_zip = wf.collect()
print("收尾包：", debug_zip)